# Azure Databricks Crypto Market Analytics & Medallion Pipeline
**Author:** Mohamed Channa (Data Science, Big Data & AI — ENSA Tétouan)

### Project Overview
This notebook implements an enterprise-grade **Medallion Architecture (Bronze -> Silver -> Gold)** on **Azure Databricks** with **PySpark** and **Delta Lake**.
It ingests 365 days of historical market data for 15 cryptocurrencies, computes advanced rolling metrics using **PySpark Window Functions**, constructs a relational **Star Schema**, and exports analytical tables for **Power BI Executive Dashboards**.

## 1. Environment Setup & Spark Session Configuration

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import os

# Initialize Databricks / Spark Session with Delta Lake support
spark = SparkSession.builder \n    .appName("CryptoMarketAnalytics") \n    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \n    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \n    .getOrCreate()

print("Spark Version:", spark.version)

## 2. Bronze Layer: Multi-Source JSON Ingestion & Flattening
Reading nested JSON payloads (, , ) from the raw CoinGecko API lake.

In [2]:
bronze_path = "data/bronze"
json_files = [f for f in os.listdir(bronze_path) if f.endswith(".json")]
print(f"Found {len(json_files)} coins in Bronze storage.")

point_schema = ArrayType(DoubleType())
chart_schema = StructType([
    StructField("prices", ArrayType(point_schema), True),
    StructField("market_caps", ArrayType(point_schema), True),
    StructField("total_volumes", ArrayType(point_schema), True)
])

silver_dfs = []
for f in json_files:
    coin_name = f.replace(".json", "")
    path = os.path.join(bronze_path, f)
    raw = spark.read.schema(chart_schema).json(path)
    
    p_df = raw.select(F.posexplode("prices").alias("pos", "p")) \n        .select(F.col("pos"), (F.col("p")[0]/1000).cast(LongType()).alias("ts"), F.col("p")[1].cast(DoubleType()).alias("price_usd"))
    m_df = raw.select(F.posexplode("market_caps").alias("pos", "m")) \n        .select(F.col("pos"), F.col("m")[1].cast(DoubleType()).alias("market_cap"))
    v_df = raw.select(F.posexplode("total_volumes").alias("pos", "v")) \n        .select(F.col("pos"), F.col("v")[1].cast(DoubleType()).alias("volume"))
        
    comb = p_df.join(m_df, "pos").join(v_df, "pos") \n        .withColumn("coin_name", F.lit(coin_name)) \n        .withColumn("Date", F.to_date(F.from_unixtime("ts"))) \n        .withColumn("ingestion_timestamp", F.current_timestamp()) \n        .select("coin_name", "price_usd", "market_cap", "volume", "Date", "ingestion_timestamp") \n        .dropDuplicates(["coin_name", "Date"])
    silver_dfs.append(comb)

silver_df = silver_dfs[0]
for df in silver_dfs[1:]:
    silver_df = silver_df.union(df)

print(f"Silver Dataset ready: {silver_df.count()} clean rows.")
silver_df.show(5)

## 3. Gold Layer: Star Schema Dimensional Modeling
Constructing  and  dimension tables with surrogate keys.

In [3]:
# dim_coin
w_coin_dim = Window.orderBy("coin_name")
dim_coin = silver_df.select("coin_name").distinct().withColumn("coin_id", F.row_number().over(w_coin_dim))
dim_coin.show()

# dim_date
dim_date = silver_df.select("Date").distinct().sort("Date") \n    .withColumn("Year", F.year("Date")) \n    .withColumn("Month", F.month("Date")) \n    .withColumn("Day", F.dayofmonth("Date")) \n    .withColumn("DayOfWeek", F.dayofweek("Date")) \n    .withColumn("is_Weekend", F.col("DayOfWeek").isin([1, 7]))
dim_date.show(5)

## 4. Gold Fact Table: Financial Metrics via PySpark Window Functions
Calculating , , , and .

In [4]:
enriched = silver_df.join(dim_coin, "coin_name", "inner")

# Define Window Specifications
w_coin = Window.partitionBy("coin_id").orderBy("Date")
w_7d = Window.partitionBy("coin_id").orderBy("Date").rowsBetween(-6, 0)
w_30d = Window.partitionBy("coin_id").orderBy("Date").rowsBetween(-29, 0)

fact_marketdata = enriched \n    .withColumn("lag_price", F.lag("price_usd", 1).over(w_coin)) \n    .withColumn("daily_return_pct", ((F.col("price_usd") - F.col("lag_price")) / F.col("lag_price")) * 100.0) \n    .withColumn("7_day_sma_price", F.avg("price_usd").over(w_7d)) \n    .withColumn("30_day_volatility", F.stddev("daily_return_pct").over(w_30d)) \n    .withColumn("7_day_sma_volume", F.avg("volume").over(w_7d)) \n    .select("coin_id", "Date", "price_usd", "market_cap", "volume", \n            "daily_return_pct", "7_day_sma_price", "30_day_volatility", "7_day_sma_volume") \n    .sort("coin_id", "Date")

fact_marketdata.show(10)

## 5. Exporting for Power BI Ingestion
Writing Star Schema tables to Delta Lake and exporting flat CSVs for Power BI Desktop.

In [5]:
# Export to Gold directory
dim_coin.toPandas().to_csv("data/gold/dim_coin.csv", index=False)
dim_date.toPandas().to_csv("data/gold/dim_date.csv", index=False)
fact_marketdata.toPandas().to_csv("data/gold/fact_marketdata.csv", index=False)
print("All Gold tables exported successfully!")